In [8]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import detectors as det
import filters as filt
import integrators as integ

csv_file_name = 'initial_walk_test_10-08-2026_16-28-57_1.csv'
csv_path = '../data/measured_walks/'
csv_save_path = '../data/orientation_mahony/'
df = pd.read_csv(f'{csv_path}{csv_file_name}')

In [9]:
# =============
# Set parameters and detect ZVWs
# =============
acc_dev = det.ACC_DEVIATION
gyro_limit = det.GYRO_LIMIT
var_limit = det.VAR_LIMIT
var_window = det.VAR_WINDOW
dwell = det.DWELL

ax = df['ax']
ay = df['ay']
az = df['az']
gx = df['gx']
gy = df['gy']
gz = df['gz']

dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0]) / 1e6
# Add a mean value at the beginning
dt_array = np.insert(dt_array, 0, dt_array.mean())

zvw_mask = det.detect_zvw(df, acc_dev, gyro_limit, var_limit, var_window, dwell)
masked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, zvw_mask, Kp=2.0, Ki=0.5)
masked_roll, masked_pitch, masked_yaw = filt.quaternions_to_euler(masked_quats)

eInt_vals = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, zvw_mask, get_eInt_vals=True)


no_zvw_mask = np.zeros_like(zvw_mask, dtype=bool)
unmasked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, no_zvw_mask, Kp=2.0, Ki=0.5)
unmasked_roll, unmasked_pitch, unmasked_yaw = filt.quaternions_to_euler(unmasked_quats)


In [10]:
# =============
# Plot Roll and pitch
# =============

time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(time_sec, unmasked_roll, label='Gyro Only (Drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax1.plot(time_sec, masked_roll, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax1.set_title('Roll (X-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax1.set_ylabel('Angle (deg)')
ax1.set_xlabel('Time (s)')
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(time_sec, unmasked_pitch, label='Gyro only (drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax2.plot(time_sec, masked_pitch, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax2.set_title('Pitch (Y-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax2.set_ylabel('Angle (deg)')
ax2.set_xlabel('Time (s)')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
# plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_roll_pitch.png'), dpi=120)
# plt.show()


In [11]:
# =============
# Plot ZVW Residuals
# =============

raw_accel = np.column_stack((ax, ay, az))

# Rotate the raw acceleration into the global frame
global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)

linear_accel = np.copy(global_accel)
linear_accel[:, 2] -= 1.0

zvs_residuals = linear_accel[zvw_mask]

mean_x_error = np.mean(zvs_residuals[:, 0])
mean_y_error = np.mean(zvs_residuals[:, 1])
mean_z_error = np.mean(zvs_residuals[:, 2])

print(f"Mean X Error: {mean_x_error} g")
print(f"Mean Y Error: {mean_y_error} g")
print(f"Mean Z Error: {mean_z_error} g")

# Histogram Z Error
fig, ax_hist = plt.subplots(1, 1, figsize=(8, 4))
ax_hist.hist(zvs_residuals[:, 0], bins=np.arange(-0.5, 0.5, 0.01), label='X Error')
ax_hist.hist(zvs_residuals[:, 1], bins=np.arange(-0.5, 0.5, 0.01), label='Y Error')
ax_hist.hist(zvs_residuals[:, 2], bins=np.arange(-0.5, 0.5, 0.01), label='Z Error')
ax_hist.set_title('Histogram of ZVW Residuals')
ax_hist.set_xlabel('ZVW Residual (g)')
ax_hist.set_ylabel('Count')
ax_hist.grid(True, linestyle='--', alpha=0.5)
ax_hist.legend()
plt.tight_layout()
# plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_ZVW_residuals.png'), dpi=120)
# plt.show()


Mean X Error: 0.00037233386285538445 g
Mean Y Error: -0.0007803020677325412 g
Mean Z Error: 0.026407278209371093 g


In [12]:
# ===========
# ZUPT
# ===========

# convert g to ms^2
accel_ms2 = linear_accel * 9.80665

# run through integrator
vel_zupt, pos_zupt, pre_zupt_vels = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)

final_xy_dist = np.linalg.norm(pos_zupt[-1, :2])
final_z_dist = pos_zupt[-1, 2]
mean_pre_zupt_x = np.mean(pre_zupt_vels[:, 0])

target_distance = 19.985

print(f"Final XY Distance: {final_xy_dist:.3f} m")
print(f"Target: 19.985 m \nError percentage: {100 - (final_xy_dist * 100 / target_distance):.3f}%")
print(f"Final Z (Vertical) Error: {final_z_dist:.3f} m")
print(f"Mean Pre-ZUPT X-Velocity: {mean_pre_zupt_x:.3f} m/s")

Final XY Distance: 19.936 m
Target: 19.985 m 
Error percentage: 0.248%
Final Z (Vertical) Error: -0.584 m
Mean Pre-ZUPT X-Velocity: 0.092 m/s


In [ ]:
# ==========================================
# Standalone 2D Trajectory Plot (Top-Down View)
# ==========================================
fig, ax_2d = plt.subplots(figsize=(10, 8))

# Plot the XY path
ax_2d.plot(pos_zupt[:, 0], pos_zupt[:, 1], label='XY Trajectory', color='tab:green', linewidth=2)

# Mark Start and End positions
ax_2d.scatter(pos_zupt[0, 0], pos_zupt[0, 1], color='blue', marker='o', s=100, label='Start', zorder=5)
ax_2d.scatter(pos_zupt[-1, 0], pos_zupt[-1, 1], color='red', marker='X', s=100, label='End', zorder=5)

ax_2d.set_title('Calculated 2D Position (Top-Down View)')
ax_2d.set_xlabel('X Position (m)')
ax_2d.set_ylabel('Y Position (m)')
ax_2d.grid(True, linestyle='--', alpha=0.5)
ax_2d.legend()
ax_2d.axis('equal')  # Keeps the X and Y scale 1:1 so the path isn't warped

plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_2D_Trajectory.png'), dpi=120)
plt.show()

In [11]:
# optimal Kp and Ki values test
kp_values = [1.0, 2.0, 4.0, 6.0, 8.0]
ki_values = [0.0, 0.1, 0.5, 1.0, 2.0]

best_score = float('inf')
best_kp = 0
best_ki = 0

raw_accel = np.column_stack((ax, ay, az))
k_pairs_values = np.zeros((25, 3))

pair_idx = 0
print("Tuning Mahony Filter on Long Walk...")
for kp in kp_values:
    for ki in ki_values:
        # Mahony with new tuning
        quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, zvw_mask, Kp=kp, Ki=ki)
        
        # Global Rotation & Gravity Removal
        global_accel = filt.rotate_vector_by_quaternion(raw_accel, quats)
        linear_accel = np.copy(global_accel)
        linear_accel[:, 2] -= 1.0
        accel_ms2 = linear_accel * 9.80665
        
        vel_zupt, pos_zupt, pre_zupt_vels = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)
        
        # Calculate Target Metric (Mean Absolute X/Y Velocity)
        mean_abs_vel = np.mean(np.abs(pre_zupt_vels[:, :2]))
        
        if mean_abs_vel < best_score:
            best_score = mean_abs_vel
            best_kp = kp
            best_ki = ki

        k_pairs_values[pair_idx] = [kp, ki, mean_abs_vel]
        pair_idx += 1

print(f"--- Tuning Complete ---")
print(f"Optimal Kp: {best_kp}")
print(f"Optimal Ki: {best_ki}")
print(f"Lowest Mean Abs Pre-ZUPT Velocity: {best_score:.4f} m/s")

Tuning Mahony Filter on Long Walk...
--- Tuning Complete ---
Optimal Kp: 2.0
Optimal Ki: 0.5
Lowest Mean Abs Pre-ZUPT Velocity: 0.1475 m/s


In [12]:

grid_scores = k_pairs_values[:, 2].reshape(len(kp_values), len(ki_values))

fig, axs1 = plt.subplots(1, 1, figsize=(8, 6))

# origin='lower' puts the first Kp value (1.0) at the bottom
im = axs1.imshow(grid_scores, cmap='viridis', origin='lower')

axs1.set_xticks(np.arange(len(ki_values)))
axs1.set_xticklabels(ki_values)
axs1.set_yticks(np.arange(len(kp_values)))
axs1.set_yticklabels(kp_values)

# annotate each cell with its exact error value
for i in range(len(kp_values)):
    for j in range(len(ki_values)):
        axs1.text(j, i, f"{grid_scores[i, j]:.3f}", 
                  ha="center", va="center", 
                  color="white" if grid_scores[i, j] < np.median(grid_scores) else "black")


# Get the matrix index positions matching your best parameters
best_row_idx = kp_values.index(best_kp)
best_col_idx = ki_values.index(best_ki)

# Draw a red star over the optimal cell
axs1.scatter(best_col_idx - 0.35, best_row_idx + 0.35, color='red', marker='*', s=250, 
             edgecolors='black', linewidths=1.2, label='Optimal Parameters', zorder=5)

axs1.set_xlabel('Ki Value')
axs1.set_ylabel('Kp Value')
axs1.set_title('Mahony Filter Tuning Error Map (Lower is Better)')
fig.colorbar(im, ax=axs1, label='Mean Abs Pre-ZUPT Velocity (m/s)')
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_Mahony_Filter_Tuning.png'), dpi=120)
plt.tight_layout()
plt.show()


In [13]:
# Match the calculated gyro bias in phase 1 with the calculated x,y bias from the Mahony filter
eInt_vals_degs = np.rad2deg(eInt_vals)
time_sec_2 = (df['t_us'] - df['t_us'].iloc[0]) / 1e6


plt.figure(figsize=(10, 5))
plt.plot(time_sec_2, eInt_vals_degs[:, 0], label=r'$K_i \cdot eInt_x$ (X Bias Est)')
plt.plot(time_sec_2, eInt_vals_degs[:, 1], label=r'$K_i \cdot eInt_y$ (Y Bias Est)')
plt.axhline(0.222, color='tab:blue', linestyle='--', alpha=0.7, label=r'Target X Bias ($+0.222^\circ$/s)')
plt.axhline(0.030, color='tab:orange', linestyle='--', alpha=0.7, label=r'Target Y Bias ($+0.030^\circ$/s)')

plt.xlabel('Time (s)')
plt.ylabel('Estimated Bias (°/s)')
plt.title('Mahony Gyroscope Bias Estimation Convergence')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
